# Kids Learning App - Database Explorer

In [2]:
import duckdb
import json
import os


def load_all_tables(family_id, kid_id=None, root_dir=None):
    """Load all tables from a kid's DB.
    Returns (table_names, dfs) where dfs[i] is the DataFrame for table_names[i].
    If kid_id is None, uses the first kid in that family.
    root_dir: data directory. Supports both live (kids.json) and backup (family_manifest.json) formats.
    """
    if root_dir is None:
        cwd = os.getcwd()
        candidates = [
            cwd,
            os.path.join(cwd, "backend", "data"),
            os.path.join(cwd, "data"),
            os.path.dirname(cwd),
            os.path.join(os.path.dirname(cwd), "backend", "data"),
        ]
        root_dir = next(
            (
                p
                for p in candidates
                if os.path.exists(os.path.join(p, "kids.json"))
                or os.path.exists(os.path.join(p, "family_manifest.json"))
            ),
            None,
        )
        if root_dir is None:
            raise FileNotFoundError(
                "Could not auto-detect data root. Pass root_dir explicitly."
            )

    # Try kids.json (live data) first, then family_manifest.json (backup)
    kids_json_path = os.path.join(root_dir, "kids.json")
    manifest_path = os.path.join(root_dir, "family_manifest.json")

    if os.path.exists(kids_json_path):
        with open(kids_json_path, "r") as f:
            metadata = json.load(f)
        all_kids = metadata.get("kids", [])
    elif os.path.exists(manifest_path):
        with open(manifest_path, "r") as f:
            metadata = json.load(f)
        all_kids = metadata.get("kids", [])
        family_id = metadata.get("family", {}).get("id", family_id)
    else:
        raise FileNotFoundError(f"No kids.json or family_manifest.json found in {root_dir}")
    
    for k in all_kids:
        print(f"{k['id']}={k['name']}, familyID={k['familyId']}")
    
    if kid_id is None:
        family_kids = [k for k in all_kids if str(k["familyId"]) == str(family_id)]
        if not family_kids:
            raise ValueError(f"No kids found for family {family_id}")
        kid_id = family_kids[0]["id"]
    
    print(f"load kid={kid_id}")

    db_path = os.path.join(root_dir, "families", f"family_{family_id}", f"kid_{kid_id}.db")
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"Kid DB not found: {db_path}")
    conn = duckdb.connect(db_path, read_only=True)

    tables = [row[0] for row in conn.execute("SHOW TABLES").fetchall()]
    dfs = [conn.execute(f"SELECT * FROM {t}").fetchdf() for t in tables]

    conn.close()
    return tables, dfs

## download

In [2]:
ROOT = "/Users/chen/Library/Mobile Documents/com~apple~CloudDocs/Downloads/"
import glob
RR = None
DD = []
for x in glob.glob(f"{ROOT}/*"):
    if x.endswith("zip"):
        continue
    RR = x
    print(RR)
    
    # From a backup/download
    def get_tables(RR):
        tables, dfs = load_all_tables(family_id=1, root_dir=RR, kid_id=2)
        for i, name in enumerate(tables):
            print(f"dfs[{i}] = {name} ({len(dfs[i])} rows)")
        return tables, dfs
    
    _, dfs = get_tables(RR)
    DD.append(dfs)

/Users/chen/Library/Mobile Documents/com~apple~CloudDocs/Downloads/session-progress-badge.png


FileNotFoundError: No kids.json or family_manifest.json found in /Users/chen/Library/Mobile Documents/com~apple~CloudDocs/Downloads/session-progress-badge.png

In [57]:
x = DD[0][0]
x[x.front.str.startswith("第")]
x[x.deck_id == 11]
y = DD[0][3]
DD[0][2]
DD[0][4]
y[y.session_id == 17]

IndexError: list index out of range

In [ ]:
dfs[0].head(3)

,id,deck_id,front,back,skip_practice,hardness_score,created_at
0,11,2,好像,好像,False,0.0,2026-02-16 06:30:57.851
1,12,2,香,香,False,0.0,2026-02-16 06:31:35.498
2,13,2,菜,菜,False,0.0,2026-02-16 06:31:58.316


# from local

In [3]:
# From local
tables, dfs = load_all_tables(family_id=1, kid_id=1)
for i, name in enumerate(tables):
    print(f"dfs[{i}] = {name} ({len(dfs[i])} rows), col={list(dfs[i].columns)}")

1=Maria, familyID=1
2=Alex, familyID=1
4=Abe, familyID=2
6=JC, familyID=3
7=Francis Mao, familyID=4
8=Eddie Mao, familyID=4
9=Matthew, familyID=5
10=Ophelia , familyID=6
14=Chen, familyID=7
15=AA, familyID=1
load kid=1
dfs[0] = cards (820 rows), col=['id', 'deck_id', 'front', 'back', 'skip_practice', 'hardness_score', 'created_at']
dfs[1] = deck_category_opt_in (7 rows), col=['category_key', 'is_opted_in', 'session_card_count', 'hard_card_percentage', 'include_orphan']
dfs[2] = decks (19 rows), col=['id', 'name', 'tags', 'created_at', 'updated_at', 'daily_target_count']
dfs[3] = kid_badge_award (48 rows), col=['award_id', 'achievement_key', 'category_key', 'badge_art_id', 'reason_text', 'evidence_json', 'awarded_at', 'celebration_seen_at']
dfs[4] = lesson_reading_audio (0 rows), col=['result_id', 'file_name', 'mime_type', 'created_at']
dfs[5] = session_results (8386 rows), col=['id', 'session_id', 'card_id', 'correct', 'response_time_ms', 'timestamp']
dfs[6] = sessions (174 rows), col=

In [4]:
dfs[6]

,id,type,planned_count,started_at,completed_at,retry_count,retry_total_response_ms,retry_best_rety_correct_count,practice_mode
0,1,chinese_characters,80,2026-02-15 23:21:27.003,2026-02-15 23:25:32.545000,0,0,0,na
1,7,chinese_characters,80,2026-02-16 15:23:27.758,2026-02-16 15:26:24.997000,0,0,0,na
2,13,chinese_characters,80,2026-02-17 22:35:21.823,2026-02-17 22:38:16.399000,0,0,0,na
3,14,chinese_characters,80,2026-02-18 22:17:42.158,2026-02-18 22:26:02.672411,0,0,0,na
4,17,chinese_characters,80,2026-02-19 23:35:12.361,2026-02-19 23:40:51.150223,0,0,0,na
...,...,...,...,...,...,...,...,...,...
169,182,chinese_characters,70,2026-04-29 21:19:37.551,2026-04-29 21:26:15.292417,0,0,0,multi
170,183,basic_math_facts,40,2026-04-29 21:26:24.121,2026-04-29 21:29:42.366484,0,0,0,multi
171,184,chinese_vocabulary,10,2026-04-29 21:29:49.746,2026-04-29 21:30:59.516679,0,0,0,multi
172,185,chinese_writing,5,2026-04-29 21:37:36.174,2026-04-29 21:39:43.014357,0,0,0,self


In [34]:
def join_card_id(dfs, idx, col_name):
    # cards
    x = dfs[0].set_index("deck_id")[['id', 'front']]
    # decks
    y = dfs[2].set_index("id").name
    
    x['deck_name'] = y
    x = x.reset_index().set_index("id")[['front', 'deck_name']]
    z = dfs[idx].set_index(col_name)
    z['front'] = x['front']
    z['deck_name'] = x['deck_name']
    return z

x = join_card_id(dfs, 0, "id")
x[x.front.isin(['香','菜','我','你'])]

,deck_id,front,back,skip_practice,hardness_score,created_at,deck_name
id,,,,,,,
8,1,香,xiāng,False,7318.0,2026-02-15 22:43:08.818,chinese_characters_orphan
173,1,我,wǒ,False,10337.0,2026-02-15 22:58:01.877,chinese_characters_orphan
276,1,你,nǐ,False,1729.0,2026-02-15 23:04:12.953,chinese_characters_orphan


In [10]:
x = dfs[3]
x[x.session_id == 13]
y = dfs[0]
y[y.id.isin([511, 512, 513])]

,id,deck_id,front,back,skip_practice,hardness_score,created_at
250,511,13,公鸡蛋（上）,50,False,0.0,2026-02-17 23:13:23.716
251,512,13,女娲造人,54,False,0.0,2026-02-17 23:13:23.728
252,513,13,叶公好龙,55,False,0.0,2026-02-17 23:13:23.737


In [40]:
# Load shared personal deck DB (global across all families)
from pathlib import Path
import duckdb

candidate_paths = [
    Path('shared_decks.duckdb'),
    Path('data/shared_decks.duckdb'),
    Path('backend/data/shared_decks.duckdb'),
    Path('../data/shared_decks.duckdb'),
]
shared_db_path = next((p.resolve() for p in candidate_paths if p.exists()), None)

if shared_db_path is None:
    raise FileNotFoundError('Could not find shared_decks.duckdb from current working directory')

conn = duckdb.connect(str(shared_db_path), read_only=True)
shared_tables = [row[0] for row in conn.execute('SHOW TABLES').fetchall()]
shared_dfs = {table: conn.execute(f'SELECT * FROM {table}').fetchdf() for table in shared_tables}
conn.close()

print(f'Loaded shared DB: {shared_db_path}')
for table in shared_tables:
    print(f'shared_dfs[\"{table}\"]: {len(shared_dfs[table])} rows')
    
shared_tables


Loaded shared DB: /Users/chen/Library/Mobile Documents/com~apple~CloudDocs/Project/backend/data/shared_decks.duckdb
shared_dfs["achievement_badge_art"]: 100 rows
shared_dfs["badge_art"]: 221 rows
shared_dfs["cards"]: 3426 rows
shared_dfs["chinese_character_bank"]: 18272 rows
shared_dfs["deck"]: 362 rows
shared_dfs["deck_category"]: 7 rows
shared_dfs["deck_generator_definition"]: 45 rows


['achievement_badge_art',
 'badge_art',
 'cards',
 'chinese_character_bank',
 'deck',
 'deck_category',
 'deck_generator_definition']

In [39]:
#shared_dfs['cards']
shared_dfs['deck_generator_definition']
x = shared_dfs['chinese_character_bank']
x[x.used]
x = shared_dfs['deck_category']
x = shared_dfs['deck']
x[x.name.str.contains("m4")]

,deck_id,name,tags,creator_family_id,created_at
